# Supply Chain Analytics - Exploratory Analysis

This notebook provides interactive analysis and visualization of supply chain KPIs.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import sys
sys.path.append('..')

from src.kpi_calculator import SupplyChainKPICalculator
from src.root_cause_analysis import RootCauseAnalyzer

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Load datasets
orders_df = pd.read_csv('../data/sample_orders.csv')
suppliers_df = pd.read_csv('../data/sample_suppliers.csv')
inventory_df = pd.read_csv('../data/sample_inventory.csv')
logistics_df = pd.read_csv('../data/sample_logistics.csv')

print(f"Orders: {len(orders_df)} records")
print(f"Suppliers: {len(suppliers_df)} records")
print(f"Inventory: {len(inventory_df)} records")
print(f"Logistics: {len(logistics_df)} records")

In [ ]:
# Preview orders data
orders_df.head()

## 2. Calculate KPIs

In [ ]:
# Initialize KPI calculator
kpi_calc = SupplyChainKPICalculator(orders_df, suppliers_df, inventory_df, logistics_df)

# Calculate all KPIs
kpi_report = kpi_calc.generate_kpi_report()

print("KPI Report Generated:", kpi_report['report_date'])

In [ ]:
# Display On-Time Delivery metrics
otd = kpi_report['on_time_delivery']
pd.DataFrame([otd])

## 3. Visualizations

In [ ]:
# On-Time Delivery Rate Visualization
otd_data = kpi_report['on_time_delivery']

fig = go.Figure(go.Indicator(
    mode = "gauge+number+delta",
    value = otd_data['otd_rate_percent'],
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': "On-Time Delivery Rate (%)"},
    delta = {'reference': 95},
    gauge = {
        'axis': {'range': [None, 100]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, 80], 'color': "lightgray"},
            {'range': [80, 95], 'color': "gray"}],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 95}}))

fig.show()

In [ ]:
# Supplier Performance Analysis
supplier_perf = pd.DataFrame.from_dict(kpi_report['supplier_performance'], orient='index')
supplier_perf = supplier_perf.sort_values('performance_score', ascending=False).head(10)

fig = px.bar(supplier_perf, 
             x=supplier_perf.index, 
             y='performance_score',
             title='Top 10 Suppliers by Performance Score',
             labels={'x': 'Supplier', 'performance_score': 'Performance Score'})
fig.show()

In [ ]:
# Inventory Turnover Metrics
inv_metrics = kpi_report['inventory_turnover']

metrics_df = pd.DataFrame([
    {'Metric': 'Inventory Turnover Ratio', 'Value': inv_metrics['inventory_turnover_ratio']},
    {'Metric': 'Days Inventory Outstanding', 'Value': inv_metrics['days_inventory_outstanding']},
    {'Metric': 'Slow Moving Items', 'Value': inv_metrics['slow_moving_items_count']}
])

fig = px.bar(metrics_df, x='Metric', y='Value', title='Inventory Metrics')
fig.show()

## 4. Root Cause Analysis

In [ ]:
# Run root cause analysis
rca = RootCauseAnalyzer(orders_df, suppliers_df, logistics_df)
rca_report = rca.generate_root_cause_report(inventory_df)

rca.print_report()

In [ ]:
# Pareto Analysis
pareto_df = rca.perform_pareto_analysis()

if not pareto_df.empty:
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=pareto_df.index,
        y=pareto_df['total_delay_days'],
        name='Total Delay Days'
    ))
    
    fig.add_trace(go.Scatter(
        x=pareto_df.index,
        y=pareto_df['cumulative_delay_percent'],
        name='Cumulative %',
        yaxis='y2'
    ))
    
    fig.update_layout(
        title='Pareto Analysis - Suppliers Contributing to Delays',
        yaxis=dict(title='Total Delay Days'),
        yaxis2=dict(title='Cumulative %', overlaying='y', side='right')
    )
    
    fig.show()

## 5. Trend Analysis

In [ ]:
# Orders trend over time
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
orders_trend = orders_df.groupby(orders_df['order_date'].dt.to_period('W')).size()

plt.figure(figsize=(12, 5))
orders_trend.plot(kind='line', marker='o')
plt.title('Orders Trend (Weekly)')
plt.xlabel('Week')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. Recommendations Summary

In [ ]:
# Compile recommendations
recommendations = []

# OTD recommendations
if otd_data['otd_rate_percent'] < 95:
    recommendations.append(f"⚠️ OTD Rate ({otd_data['otd_rate_percent']}%) is below target (95%). Focus on supplier performance improvement.")

# Inventory recommendations
inv_recs = rca_report['inventory_analysis']['recommendations']
recommendations.extend([f"📦 {rec}" for rec in inv_recs])

# Bottleneck recommendations
bottleneck_recs = rca_report['procurement_bottlenecks']['recommendation']
recommendations.extend([f"🔧 {rec}" for rec in bottleneck_recs])

print("\n=== ACTION RECOMMENDATIONS ===")
for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec}")